# When the context fills, keep the evidence
## Persistent notes and searchable history: Oracle AI Database edition

**Download this notebook, open it in Jupyter, and run it from top to bottom.**
Everything we implement is in these cells. Installation uses the published
`memorizz` package; no repository checkout or companion Python file is needed.

We follow a debugging task through three context windows using real
**GPT-6 Astra** responses and **text-embedding-3-small** embeddings. A failure
disappears from the active prompt and is absent from the notes, but the agent
can recover it from the original tool output.

Allow 45–60 minutes to read and experiment. Each code cell has at most 25 lines;
the surrounding explanations introduce one piece at a time. Running the notebook
makes paid OpenAI API calls. The Oracle edition also needs a reachable database.

## 1 · The story: a fix fails, the summary survives, the reason disappears

Imagine an agent debugging retry behavior in a client library. Early in the
session, the user says that attempt order must remain stable. A first fix changes
the delay calculation. A regression test still fails, reporting an expected
delay, an actual delay, and a clue about clock skew.

Several investigations later, the context window fills. A plausible summary
reads: *“Investigated retry timing; tests still fail.”* That sentence helps the
agent stay oriented. It cannot answer which test failed, what values it reported,
or why repeating the same fix would waste time. When that summary is the only
retained record, the omitted evidence is difficult to recover reliably.

The memory-management paragraph in OpenAI's
[GPT-6 Astra announcement](https://openai.com/index/gpt-6-astra/)
describes two complementary capabilities in Codex: accumulated notes can survive
across context windows, and earlier messages and tool outputs remain searchable.
The second capability matters because notes cannot anticipate every later question.

We use the supplied paragraph as a behavioral specification. We have not
inspected Codex's private implementation or measured its memory system. It does
not disclose the note schema, writing policy, ranking algorithm, or rollover
threshold. Nor does it establish that summaries are never used. Our implementation
makes those application choices explicit so students can inspect and change them.

**The design lesson:** preserve source evidence, maintain conclusions individually,
and choose separately what belongs in the next model request.

### What is described, observed, and implemented

| Basis | What we can say |
|---|---|
| Supplied OpenAI paragraph | Notes persist across windows; earlier messages and tool outputs remain searchable, including details absent from notes. |
| Published Memorizz 0.10.0 | Filesystem and Oracle providers offer storage, scoped retrieval, embeddings, and the OpenAI Responses adapter. |
| This notebook | A small note journal, complete source records, a derived chunk index, explicit turn capture, and stored window checkpoints. |
| Our exercise | The first notes deliberately omit one diagnostic. GPT-6 must search and read its source before writing a next-step note. |
| Still unknown | Codex's internal formats, training, algorithms, triggers, concurrency rules, and performance. |

This demonstrates the described behavior on a constructed task. It does not
establish algorithmic equivalence or quality parity with Codex.

## 2 · Reference architecture

Think of a **notebook of conclusions**, a **filing cabinet of evidence**, and a
**desk holding today's work**. Notes belong in the notebook; complete tool outputs
belong in the cabinet; selected notes and source excerpts go on the desk.

```mermaid
flowchart TD
    User[User request] --> Host[Notebook host and current thread]
    Host --> Agent[MemAgent]
    Agent <--> LLM[OpenAI GPT-6 Astra via Responses]
    Agent --> Work[Work tools capture full results before returning]
    Host --> Archive[Complete user and assistant turns]
    Work --> Archive
    Archive --> Sources[Private SHARED_MEMORY source records]
    Archive --> Chunks[Exact source chunks and offsets]
    Chunks --> Embed[OpenAI text-embedding-3-small]
    Embed --> Index[KNOWLEDGE_BASE vector index]
    Host --> Notes[Private SHARED_MEMORY note revisions]
    Host --> State[Private SHARED_MEMORY checkpoint]
    Notes --> Bootstrap[Bounded note bootstrap]
    Bootstrap --> Fresh[Fresh thread for the same task]
    State --> Fresh
    Fresh --> Search[Search history]
    Search --> Index
    Index --> Read[Read original source by ID and offset]
    Sources --> Read
    Read --> Fresh
```

All memory boxes use the **Oracle AI Database provider**. OpenAI supplies generation and
embeddings. `SHARED_MEMORY` is the package's record-storage partition; our
application wraps each record with an explicit task/user scope and marks it
private. The host must bind those IDs to its authenticated user in a real app.

Notes use exact record lookups. Only source chunks need semantic ranking here.
This also avoids relying on unreleased Oracle knowledge-base read-by-ID changes.
The source archive is canonical; the vector index can be rebuilt from it.
Mermaid diagrams render in compatible viewers and remain readable as Markdown.

## 3 · Install the published package

Use Python 3.10 or later in a fresh Jupyter kernel. The cell below runs `pip`
inside that kernel. We pin the **published 0.10.0 release** so the lesson has a
repeatable API surface; this is a normal PyPI installation. The base command is
`pip install memorizz`; the Oracle edition adds the package's `oracle` extra.

If you need Jupyter itself, install it in your environment with
`python -m pip install jupyterlab`, then open this downloaded notebook.
If updating a package already imported in your kernel, restart the kernel before
continuing. Run all cells in order; a fresh run creates a fresh task scope.

In [1]:
%pip install -q "memorizz[oracle]==0.10.0"


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import logging
import os
import re
import tempfile
import uuid
from copy import deepcopy
from getpass import getpass
from importlib.metadata import version
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

logging.getLogger("memorizz").setLevel(logging.ERROR)
print({"memorizz": version("memorizz"), "openai": version("openai")})

{'memorizz': '0.10.0', 'openai': '3.14.0'}


### Credentials stay outside the notebook

Set `OPENAI_API_KEY` in the environment or place a private `.env` beside this
notebook. If absent, the next cell requests it using hidden input. The same key
is used for GPT-6 and embeddings; it must have access to both models. Never paste
a key into a code cell or save it in outputs. Errors stop execution rather than
substituting another model or provider.

In [3]:
load_dotenv(Path.cwd() / ".env", override=False)
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ").strip()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is required.")


def show(value):
    display(Markdown("```json\n" + json.dumps(value, indent=2) + "\n```"))


RUN_ID = uuid.uuid4().hex[:12]
MEMORY_ID, USER_ID = f"context-{RUN_ID}", f"student-{RUN_ID}"
KEEP_DATA = os.getenv("MEMORIZZ_CONTEXT_KEEP_DATA", "0") == "1"

### One embedding model for both indexing and querying

`text-embedding-3-small` supports shortening vectors through the
[`dimensions` parameter](https://developers.openai.com/api/docs/guides/embeddings#reducing-embedding-dimensions).
We request **384 dimensions**, matching the prepared Oracle schema used for this
lesson. Both providers receive this same embedding manager, and every archive
chunk is embedded with it. Change dimensions only to match your database schema
and rebuild the index in a fresh scope. Equal vector lengths do not make vectors
from different models comparable.

In [4]:
from memorizz.embeddings import EmbeddingManager, set_global_embedding_manager

DIMENSIONS = int(os.getenv("MEMORIZZ_CONTEXT_DIMENSIONS", "384"))
if not 1 <= DIMENSIONS <= 1536:
    raise ValueError("Choose 1–1536 dimensions for text-embedding-3-small.")
embeddings = EmbeddingManager(
    provider="openai",
    config={"model": "text-embedding-3-small", "dimensions": DIMENSIONS},
)
set_global_embedding_manager(embeddings)
probe = embeddings.get_embedding("An earlier retry-delay fix failed.")
assert len(probe) == DIMENSIONS and any(probe)
show({"embedding_model": "text-embedding-3-small", "dimensions": len(probe)})

```json
{
  "embedding_model": "text-embedding-3-small",
  "dimensions": 384
}
```

### Connect to Oracle AI Database

Use an existing development database with a Memorizz-compatible schema and VECTOR
columns of the configured dimension. Supply `ORACLE_USER`, `ORACLE_PASSWORD`, and
`ORACLE_DSN` in the environment or local `.env`; missing values are requested
below. A DSN looks like `localhost:1521/FREEPDB1`.

This notebook does not provision a database server. Provider construction can
initialize missing Memorizz tables in the chosen schema. Your database user needs
the corresponding schema permissions. Preflight and vector-dimension validation
must pass before the lesson writes its task records.

`index_policy="none"` uses exact vector-distance retrieval for this small dataset.
OpenAI generates the vectors; `in_database_embedding=False` selects that path.

In [5]:
from memorizz import OracleProvider

for key in ("ORACLE_USER", "ORACLE_PASSWORD", "ORACLE_DSN"):
    if not os.getenv(key):
        os.environ[key] = getpass(f"{key}: ").strip()
    if not os.getenv(key):
        raise RuntimeError(f"{key} is required.")


def open_provider():
    return OracleProvider.from_env(
        index_policy="none",
        in_database_embedding=False,
        embedding_provider=embeddings,
    )


provider = open_provider()

In [6]:
preflight = provider.preflight()
show(
    {
        key: preflight.get(key)
        for key in ("ok", "database_product", "version_full", "index_policy")
    }
)
if not preflight.get("ok"):
    raise RuntimeError("Oracle preflight failed; inspect the connection and schema.")
provider.validate_vector_schema_dimensions(DIMENSIONS)
assert provider.__class__.__name__ == "OracleProvider"

```json
{
  "ok": true,
  "database_product": "Oracle AI Database 26ai Free",
  "version_full": "23.26.0.0.0",
  "index_policy": "none"
}
```

## 4 · Build a small persistent journal

The following short cells define the complete memory implementation. They use
published provider methods: `store`, `retrieve_by_id`, `search_memory`, and
`delete_by_id`. This is notebook application code, not a new Memorizz API.

Our session holds a provider, an embedding manager, an explicit scope, and a
checkpoint. Stable IDs make retries address the same source. A SHA-256 digest
detects accidental changes to archived content; it is not an authentication
mechanism. The session is designed for one writer.

In [7]:
import hashlib
import json
import uuid
from copy import deepcopy
from dataclasses import dataclass

from memorizz import MemoryType


def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False)


def digest(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def record_id(row):
    return str(row.get("id") or row.get("_id"))

In [8]:
@dataclass
class MemorySession:
    provider: object
    embeddings: object
    scope: dict
    state_id: str
    state: dict


def stable_id(session, *parts):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, canonical([session.scope, *parts])))


def run_scope(session):
    return {**session.scope, "thread_id": session.state["window"]}

### Store and read scoped records

Each private record contains a format marker, task/user scope, and payload.
Reads validate those fields even when the caller already knows a record ID.
The logical `memory_id` addresses a shared-memory record on both providers.
Oracle's physical deletion ID is handled separately in the cleanup section.

In [9]:
def write_record(session, identifier, value):
    envelope = {"format": "context-notebook-v1", "scope": session.scope, "value": value}
    stored = session.provider.store(
        {
            "_id": identifier,
            "memory_id": identifier,
            "content": canonical(envelope),
            "scope": "private",
            "owner_agent_id": session.state_id,
            "record_type": "context_notebook",
        },
        MemoryType.SHARED_MEMORY,
    )
    if not stored:
        raise RuntimeError("Memory write failed.")

In [10]:
def read_record(session, identifier):
    row = session.provider.retrieve_by_id(identifier, MemoryType.SHARED_MEMORY)
    if row is None:
        return None
    envelope = json.loads(row["content"])
    if (
        envelope.get("format") != "context-notebook-v1"
        or envelope.get("scope") != session.scope
    ):
        raise ValueError("Record is outside this task/user scope.")
    return envelope["value"]


def save_state(session):
    write_record(session, session.state_id, session.state)

### Reopen from a stored checkpoint

The checkpoint lists source IDs, note revisions, index IDs, and completed windows.
A reopened session reads that record rather than receiving the previous session's
Python dictionary. We save a checkpoint only after the records it references have
been written. This ordering helps retries but is not a multi-record transaction.

In [11]:
def open_session(provider, embeddings, memory_id, user_id):
    if not memory_id or not user_id:
        raise ValueError("Explicit task and user IDs are required.")
    scope = {"memory_id": memory_id, "user_id": user_id}
    session = MemorySession(provider, embeddings, scope, "", {})
    session.state_id = stable_id(session, "state")
    session.state = read_record(session, session.state_id) or {
        "window": "window-1",
        "windows": ["window-1"],
        "sources": [],
        "notes": {},
        "index_ids": [],
        "checkpoints": [],
    }
    save_state(session)
    return session

### Derive searchable chunks from complete sources

We use overlapping character slices: 900 characters per chunk, starting every
700 characters. Each chunk stores its original source ID, offset, and digest.
These choices keep the example small and the offsets exact; they are not claimed
to be Codex's indexing algorithm. Only the chunk vectors enter similarity search.

In [12]:
def chunk_record(session, source, start):
    piece = source["content"][start : start + 900]
    return {
        "_id": stable_id(session, "chunk", source["source_id"], start),
        **session.scope,
        "namespace": session.state_id,
        "content": piece,
        "embedding": session.embeddings.get_embedding(piece),
        "metadata": {
            "source_id": source["source_id"],
            "start": start,
            "sha256": source["sha256"],
        },
    }


def index_source(session, source):
    for start in range(0, max(1, len(source["content"])), 700):
        identifier = stable_id(session, "chunk", source["source_id"], start)
        if not session.provider.retrieve_by_id(identifier, MemoryType.KNOWLEDGE_BASE):
            row = chunk_record(session, source, start)
            if not session.provider.store(row, MemoryType.KNOWLEDGE_BASE):
                raise RuntimeError("Index write failed.")
        if identifier not in session.state["index_ids"]:
            session.state["index_ids"].append(identifier)

In [13]:
def archive_source(session, content, kind, origin):
    identifier = stable_id(session, "source", kind, origin)
    source = {
        "source_id": identifier,
        "kind": kind,
        "origin": origin,
        "window": session.state["window"],
        "content": content,
        "sha256": digest(content),
    }
    existing = read_record(session, identifier)
    if existing and existing != source:
        raise ValueError("An archived source is immutable; origin was reused.")
    if not existing:
        write_record(session, identifier, source)
    index_source(session, source)
    if identifier not in session.state["sources"]:
        session.state["sources"].append(identifier)
    save_state(session)
    return identifier

### Read evidence in bounded slices

Search will return excerpts, but a model can open the underlying source for
details. A read verifies scope, membership in the checkpoint, and the source
digest. `has_more` tells the caller when another slice is available. Character
budgets make the example inspectable; a production host should also count tokens.

In [14]:
def read_source(session, source_id, start=0, max_chars=1600):
    if source_id not in session.state["sources"]:
        raise ValueError("Unknown source in this task.")
    if type(start) is not int or start < 0 or not 1 <= max_chars <= 8000:
        raise ValueError("Invalid source slice.")
    source = read_record(session, source_id)
    if source is None or digest(source["content"]) != source["sha256"]:
        raise ValueError("Archived source is missing or changed.")
    text = source["content"]
    return {
        **source,
        "content": text[start : start + max_chars],
        "start": start,
        "total_chars": len(text),
        "has_more": start + max_chars < len(text),
    }

### Search, then validate the returned evidence

Both providers generate a query vector with the configured OpenAI embedding
manager and rank stored chunks. We recheck candidate scope and resolve every hit
against the canonical archive. The final result has bounded size and at most one
excerpt per source. A semantic score is a ranking signal, not proof of relevance.

In [15]:
def source_hit(session, candidate):
    if (
        any(candidate.get(k) != v for k, v in session.scope.items())
        or candidate.get("namespace") != session.state_id
    ):
        raise ValueError("Search returned an out-of-scope chunk.")
    meta = candidate["metadata"]
    meta = json.loads(meta) if isinstance(meta, str) else meta
    source = read_source(session, meta["source_id"], meta["start"], 600)
    if source["sha256"] != meta["sha256"]:
        raise ValueError("Index and archive disagree.")
    return {
        "source_id": source["source_id"],
        "window": source["window"],
        "kind": source["kind"],
        "start": meta["start"],
        "excerpt": source["content"],
        "score": candidate.get("score"),
    }

In [16]:
def search_history(session, query, limit=4, max_chars=3000):
    if not query.strip():
        return []
    if not 1 <= limit <= 10 or not 200 <= max_chars <= 8000:
        raise ValueError("Search output must be bounded.")
    candidates = session.provider.search_memory(
        query,
        MemoryType.KNOWLEDGE_BASE,
        limit=20,
        namespace=session.state_id,
        **session.scope,
    )
    hits, seen = [], set()
    for candidate in candidates:
        hit = source_hit(session, candidate)
        if hit["source_id"] in seen or len(canonical([*hits, hit])) > max_chars:
            continue
        hits.append(hit)
        seen.add(hit["source_id"])
        if len(hits) == limit:
            break
    return hits

### Revise one note at a time

A note records a conclusion, category, revision, previous revision ID, and source
references. Revisions are immutable records; the checkpoint selects the current
one. Revising the API requirement will not rewrite the failed-attempt note.

The expected revision rejects stale updates within this single-writer lesson.
Source validation proves that the cited evidence exists in this task. It does
not prove that a conclusion correctly interprets that evidence.

In [17]:
def validate_note(session, key, content, category, source_ids, expected_revision):
    revisions = session.state["notes"].get(key, [])
    if expected_revision != len(revisions):
        raise ValueError("Note revision conflict; read the current note first.")
    if category not in {
        "requirement",
        "decision",
        "failed_attempt",
        "behavior",
        "next_step",
    }:
        raise ValueError("Unknown note category.")
    if not key.strip() or not content.strip() or len(content) > 1800 or not source_ids:
        raise ValueError("A bounded note and supporting sources are required.")
    for identifier in source_ids:
        read_source(session, identifier, max_chars=1)
    return revisions

In [18]:
def write_note(session, key, content, category, source_ids, expected_revision=0):
    revisions = validate_note(
        session, key, content, category, source_ids, expected_revision
    )
    revision = len(revisions) + 1
    identifier = stable_id(session, "note", key, revision)
    note = {
        "note_id": identifier,
        "key": key,
        "content": content,
        "category": category,
        "revision": revision,
        "supersedes": revisions[-1] if revisions else None,
        "source_ids": source_ids,
        "window": session.state["window"],
    }
    existing = read_record(session, identifier)
    if existing and existing != note:
        raise ValueError("Immutable note revision has different content.")
    write_record(session, identifier, note)
    session.state["notes"][key] = [*revisions, identifier]
    save_state(session)
    return note

In [19]:
def read_notes(session, include_history=False):
    notes = []
    for key, ids in sorted(session.state["notes"].items()):
        for identifier in ids if include_history else ids[-1:]:
            note = read_record(session, identifier)
            if note is None or note["key"] != key:
                raise ValueError("Note is missing or inconsistent.")
            notes.append(note)
    return notes


def bootstrap(session, max_chars=5000):
    value = {"memory_notes": read_notes(session), "window": session.state["window"]}
    if len(canonical(value)) > max_chars:
        raise ValueError("Notes exceed the bootstrap budget; explicitly select notes.")
    return value

## 5 · A boundary closes the window, not the task

```mermaid
sequenceDiagram
    participant H as Notebook host
    participant A as Current MemAgent
    participant D as Memory provider
    participant N as Fresh MemAgent
    H->>A: Complete current turn
    A->>D: Tools capture full results before returning
    H->>D: Archive user and final assistant messages
    H->>D: Save individual notes and window checkpoint
    H->>N: Start a new thread for the same task and user
    D-->>N: Bounded current-note bootstrap
    N->>D: Search earlier sources
    D-->>N: Source IDs and excerpts
    N->>D: Read selected source
    D-->>N: Exact evidence and provenance
```

We force boundaries between completed turns. A production host could trigger
the same sequence after a prompt-budget check. The checkpoint does not expand
the model's context window or preserve hidden reasoning. New threads under the
same task/user scope make it easy to inspect what survives.

In [20]:
def rollover(session, next_window):
    if not next_window or next_window in session.state["windows"]:
        raise ValueError("Choose a new window identifier.")
    checkpoint = stable_id(session, "window", session.state["window"])
    write_record(session, checkpoint, deepcopy(session.state))
    session.state["checkpoints"].append(checkpoint)
    session.state["windows"].append(next_window)
    session.state["window"] = next_window
    save_state(session)
    return {"checkpoint_id": checkpoint, "window": next_window}

### Capture at the point where evidence exists

The published 0.10.0 release lacks a `persist_all_results` policy and a direct
`search_tool_logs` helper. We therefore capture full results in the notebook's
work tools **before** returning
them to MemAgent. Short failures and long logs both enter the archive. MemAgent
can still offload long results in its active prompt.

The host wrapper also archives each user request and final assistant response.
It does not reconstruct every intermediate message or supplied context payload.
Tool reads that merely expand existing evidence are not archived recursively.

In [21]:
def capture_tool_result(session, name, result):
    payload = canonical(
        {"tool_name": name, "arguments": {}, "result": result, "error": None}
    )
    archive_source(
        session, payload, "tool", {"tool_name": name, "call_id": uuid.uuid4().hex}
    )
    return result


def run_turn(session, agent, prompt, context=None):
    turn_id = uuid.uuid4().hex
    user_source = archive_source(session, prompt, "user", {"turn_id": turn_id})
    answer = agent.run(prompt, context=context, **run_scope(session))
    archive_source(session, answer, "assistant", {"turn_id": turn_id})
    return answer, user_source

## 6 · Connect real GPT-6 tool use

The [OpenAI model guide](https://developers.openai.com/api/docs/guides/latest-model)
specifies `gpt-6-astra`, the Responses API for function tools, and reasoning
effort starting at `low`. We use a 4,096-token output limit per request and set
the documented context size explicitly because this package release predates
GPT-6's model-name mapping. A tool loop may make several requests.

The observer below delegates every generation to Memorizz's real OpenAI adapter.
It records input messages and actual tool choices for our checks; it does not
choose tools, generate answers, or invent usage. These diagnostics stay in memory.

In [22]:
from memorizz import ContextPolicy, MemAgent, OpenAI, RetrievalPolicy, ToolResultPolicy


def tool_calls_from(response):
    calls = []
    for choice in getattr(response, "choices", []) or []:
        for call in getattr(choice.message, "tool_calls", []) or []:
            calls.append(
                {"name": call.function.name, "arguments": call.function.arguments}
            )
    return calls


class RecordedOpenAI(OpenAI):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.requests, self.calls = [], []

    def generate(self, messages, tools=None, tool_choice="auto"):
        self.requests.append(deepcopy(messages))
        response = super().generate(messages, tools, tool_choice)
        self.calls.extend(tool_calls_from(response))
        return response

In [23]:
def new_model():
    return RecordedOpenAI(
        model="gpt-6-astra",
        api_mode="responses",
        reasoning_effort="low",
        max_completion_tokens=4096,
        context_window_tokens=1_050_000,
    )

### Give tools a bound memory session

The factories below close over one session, so the model does not supply a task
or user ID. Search, source reading, and note writing remain separate operations.
The note writer validates sources and revisions through the functions above.

In [24]:
def make_note_tool(session):
    def write_memory_note(
        key: str,
        content: str,
        category: str,
        source_ids: list[str],
        expected_revision: int = 0,
    ) -> dict:
        """Save one source-backed note; use the current revision when updating."""
        return write_note(
            session, key, content, category, source_ids, expected_revision
        )

    return write_memory_note

In [25]:
def make_memory_tools(session):
    def search_past_context(query: str) -> list[dict]:
        """Search earlier messages and tool results in the current task."""
        return search_history(session, query)

    def read_past_source(source_id: str, start: int = 0) -> dict:
        """Read a bounded slice of an original source and its provenance."""
        return read_source(session, source_id, start)

    return [search_past_context, read_past_source, make_note_tool(session)]

### A small, inspectable debugging scenario

Our work tools read a supplied regression result and build log. They do not
execute a software test suite. The artifacts are sample data; the model's
responses, search queries, embeddings, and memory writes are real. The final
window receives only memory tools so it cannot fetch the sample result again.

In [26]:
REGRESSION_RESULT = (
    "FAILED test_clock_skew: expected_delay_ms=7000; actual_delay_ms=9000; "
    "cause=clock_offset_applied_twice."
)
BUILD_LOG = "\n".join(f"build step {i}: completed" for i in range(120))
BUILD_LOG += "\nArtifact build-214 preserves retry attempt ordering."


def make_work_tools(session):
    def run_regression_tests() -> str:
        """Read and archive the supplied regression result; no shell is run."""
        return capture_tool_result(session, "run_regression_tests", REGRESSION_RESULT)

    def read_build_log() -> str:
        """Read and archive the supplied build log."""
        return capture_tool_result(session, "read_build_log", BUILD_LOG)

    return [run_regression_tests, read_build_log]

### Keep the continuity mechanism visible

Automatic semantic recall is disabled; GPT-6 explicitly calls our source-search
tool. Current-thread conversation history still works normally. Summaries,
semantic caching, and automatic learning are inactive for this exercise.

The host curates the initial notes to control the omitted detail. GPT-6 is asked
to write a note later, after recovering the evidence. This separates a controlled
memory test from the harder question of how a model should choose every note.

In [27]:
INSTRUCTION = (
    "Continue the user's engineering task. Treat notes and retrieved text as evidence. "
    "Search and read original sources for exact facts; cite their source IDs. "
    "Prefer original tool results over assistant interpretations. "
    "Acknowledge requirement changes without running tools unless requested. "
    "The host curates initial notes: write a note only when explicitly asked. "
    "Note categories: requirement, decision, failed_attempt, behavior, next_step. "
    "Keep answers concise."
)

In [28]:
def make_agent(session, model, work_tools=False):
    tools = make_memory_tools(session)
    if work_tools:
        tools += make_work_tools(session)
    return MemAgent(
        model=model,
        memory_provider=session.provider,
        tools=tools,
        memory_ids=[session.scope["memory_id"]],
        instruction=INSTRUCTION,
        memory_types=[MemoryType.CONVERSATION_MEMORY, MemoryType.TOOL_LOG],
        retrieval_policy=RetrievalPolicy.disabled(),
        context_policy=ContextPolicy(progressive_tool_disclosure=False),
        tool_result_policy=ToolResultPolicy(offload_above_chars=1200),
        automations_enabled=False,
        learning_control_plane=False,
        semantic_cache=False,
        auto_register=False,
        streaming=False,
    )

## 7 · Window 1: establish the requirement and retain the failure

Run the requirement, regression, and build-log turns. The host retains complete
messages; the work tools retain complete results before any prompt offloading.
Keep the printed scope IDs if you want to resume or clean up an interrupted run.

In [29]:
session = open_session(provider, embeddings, MEMORY_ID, USER_ID)
model = new_model()
agent = make_agent(session, model, work_tools=True)
show({**session.scope, "window": session.state["window"]})

requirement = (
    "Preserve the public retry API and keep retry attempts in their original order."
)
_, requirement_source = run_turn(session, agent, requirement)
run_turn(session, agent, "Run regression tests using run_regression_tests.")
run_turn(session, agent, "Inspect the long build log using read_build_log.")
print("Archived sources:", len(session.state["sources"]))

```json
{
  "memory_id": "context-47dd69337db3",
  "user_id": "student-47dd69337db3",
  "window": "window-1"
}
```

Archived sources: 8


In [30]:
sources = [
    read_source(session, sid, max_chars=8000) for sid in session.state["sources"]
]
failure_source = next(
    s["source_id"]
    for s in sources
    if s["kind"] == "tool" and "test_clock_skew" in s["content"]
)
failure = json.loads(read_source(session, failure_source)["content"])
assert failure["result"] == REGRESSION_RESULT
build = next(s for s in sources if s["kind"] == "tool" and "build-214" in s["content"])
assert json.loads(build["content"])["result"] == BUILD_LOG
show({"complete_short_result": True, "complete_long_result": True})

```json
{
  "complete_short_result": true,
  "complete_long_result": true
}
```

### Deliberately omit the exact diagnostic from the initial notes

The first note retains the user's requirement. The second records that the
first fix failed and that its original evidence should be consulted. It omits
the test name, numeric values, and diagnostic cause. This controlled omission
lets us ask whether searchable evidence adds something the notes alone lack.

In [31]:
write_note(session, "api-contract", requirement, "requirement", [requirement_source])
write_note(
    session,
    "first-fix",
    "The first retry-delay fix failed regression testing. Read the original result before repeating it.",
    "failed_attempt",
    [failure_source],
)
initial_notes = read_notes(session)
assert "test_clock_skew" not in canonical(initial_notes)
assert "clock_offset_applied_twice" not in canonical(initial_notes)
show(initial_notes)

```json
[
  {
    "category": "requirement",
    "content": "Preserve the public retry API and keep retry attempts in their original order.",
    "key": "api-contract",
    "note_id": "fcad9c77-4a68-53fe-9af8-33b443856a5e",
    "revision": 1,
    "source_ids": [
      "4ffd2826-7aaf-5153-a751-ae0e6d8912e1"
    ],
    "supersedes": null,
    "window": "window-1"
  },
  {
    "category": "failed_attempt",
    "content": "The first retry-delay fix failed regression testing. Read the original result before repeating it.",
    "key": "first-fix",
    "note_id": "1eadb865-3f06-51ad-8505-1f36f40d766f",
    "revision": 1,
    "source_ids": [
      "b67f6260-1709-5450-a42e-e16e00b54c64"
    ],
    "supersedes": null,
    "window": "window-1"
  }
]
```

## 8 · Window 2: incorporate a changed requirement

A new thread gets only the note bootstrap and the new instruction. The original
test output stays out of its first request. We revise just the API-contract note
to add a four-attempt limit. The first revision remains readable, and the
failed-attempt note retains its identity and revision.

In [32]:
agent.close(close_memory_provider=False)
model.client.close()
show(rollover(session, "window-2"))
model = new_model()
agent = make_agent(session, model)
steering = "Add a maximum of four retry attempts while preserving the existing API and attempt order."
_, steering_source = run_turn(session, agent, steering, bootstrap(session))
assert "test_clock_skew" not in canonical(model.requests[0])
assert "clock_offset_applied_twice" not in canonical(model.requests[0])

```json
{
  "checkpoint_id": "8a584ae0-8790-5aac-8261-28e578522f73",
  "window": "window-2"
}
```

In [33]:
write_note(
    session,
    "api-contract",
    steering,
    "requirement",
    [requirement_source, steering_source],
    expected_revision=1,
)
versions = [n for n in read_notes(session, True) if n["key"] == "api-contract"]
assert [n["revision"] for n in versions] == [1, 2]
assert versions[1]["supersedes"] == versions[0]["note_id"]
assert versions[0]["content"] == requirement
assert next(n for n in read_notes(session) if n["key"] == "first-fix")["revision"] == 1
show(versions)

```json
[
  {
    "category": "requirement",
    "content": "Preserve the public retry API and keep retry attempts in their original order.",
    "key": "api-contract",
    "note_id": "fcad9c77-4a68-53fe-9af8-33b443856a5e",
    "revision": 1,
    "source_ids": [
      "4ffd2826-7aaf-5153-a751-ae0e6d8912e1"
    ],
    "supersedes": null,
    "window": "window-1"
  },
  {
    "category": "requirement",
    "content": "Add a maximum of four retry attempts while preserving the existing API and attempt order.",
    "key": "api-contract",
    "note_id": "31d15fdf-0e9d-5e6b-b840-9b7a9f9bc745",
    "revision": 2,
    "source_ids": [
      "4ffd2826-7aaf-5153-a751-ae0e6d8912e1",
      "e4a197d3-3d99-50d6-8e5e-a39166d16d7d"
    ],
    "supersedes": "fcad9c77-4a68-53fe-9af8-33b443856a5e",
    "window": "window-2"
  }
]
```

## 9 · Close the provider, then recover the stored task

The reopen check creates a new provider and session. The checkpoint and note
revisions must come back from storage. The sample data still exists in this
kernel, but the next agent has only memory tools. It must retrieve the earlier
failure rather than invoking the regression reader again.

In [34]:
agent.close(close_memory_provider=False)
model.client.close()
show(rollover(session, "window-3"))
expected_notes = read_notes(session)
expected_sources = list(session.state["sources"])
provider.close()

provider = open_provider()
session = open_session(provider, embeddings, MEMORY_ID, USER_ID)
assert session.state["window"] == "window-3"
assert read_notes(session) == expected_notes
assert session.state["sources"] == expected_sources
show({"notes_survived": True, "sources_survived": len(expected_sources)})

```json
{
  "checkpoint_id": "869984ed-298d-5ab4-a4f2-ba39fb367917",
  "window": "window-3"
}
```

```json
{
  "notes_survived": true,
  "sources_survived": 10
}
```

### Ask a question the notes cannot answer

The question below supplies neither the test identifier nor the delay values.
GPT-6 chooses a semantic search query, opens the original tool source, and cites
it. We inspect actual tool calls as well as the final answer. Wording and query
choices can vary between runs; a failed assertion is a result to inspect, not a
reason to replace the model's response with a canned answer.

In [35]:
model = new_model()
agent = make_agent(session, model)
question = (
    "Why did the earlier retry-delay fix fail? Use search_past_context and "
    "read_past_source to inspect its original regression tool output. Report "
    "the exact test name, expected and actual delays in milliseconds, and the "
    "diagnostic cause identifier. Cite that original tool source ID."
)
answer, _ = run_turn(session, agent, question, bootstrap(session))
show(
    {
        "answer": answer,
        "actual_tool_calls": model.calls,
        "last_api_usage": model.get_last_usage(),
    }
)

```json
{
  "answer": "The earlier fix failed because the clock offset was applied twice.\n\n- **Test:** `test_clock_skew`\n- **Expected delay:** `7000 ms`\n- **Actual delay:** `9000 ms`\n- **Diagnostic cause:** `clock_offset_applied_twice`\n\nSource: original `run_regression_tests` output, `b67f6260-1709-5450-a42e-e16e00b54c64`.",
  "actual_tool_calls": [
    {
      "name": "search_past_context",
      "arguments": "{\"query\":\"earlier retry-delay fix failed regression expected actual delay diagnostic cause\"}"
    },
    {
      "name": "read_past_source",
      "arguments": "{\"source_id\":\"b67f6260-1709-5450-a42e-e16e00b54c64\",\"start\":0}"
    }
  ],
  "last_api_usage": {
    "prompt_tokens": 4501,
    "completion_tokens": 94,
    "total_tokens": 4595,
    "cached_tokens": 4257,
    "cache_write_tokens": 241,
    "reasoning_tokens": 0
  }
}
```

In [36]:
assert "expected_delay_ms=7000" not in canonical(model.requests[0])
assert "clock_offset_applied_twice" not in canonical(model.requests[0])
assert any(c["name"] == "search_past_context" for c in model.calls)
assert any(
    c["name"] == "read_past_source"
    and json.loads(c["arguments"])["source_id"] == failure_source
    for c in model.calls
)
assert "test_clock_skew" in answer and "clock_offset_applied_twice" in answer
assert re.search(r"\b7,?000\b", answer) and re.search(r"\b9,?000\b", answer)
assert failure_source in answer
print("GPT-6 searched, read the original source, and cited the missing facts.")

GPT-6 searched, read the original source, and cited the missing facts.


## 10 · Check the boundaries of the demonstration

This comparison uses an intentionally incomplete summary. It is an omission
exercise, not a benchmark of a real summarizer or OpenAI compaction. We also
check bounded retrieval and scope rejection. A known source ID must not grant
access to a different task or user.

In [37]:
summary_only = "Investigated retry timing; the first fix failed regression testing."
assert "expected_delay_ms=7000" not in summary_only
assert "expected_delay_ms=7000" not in canonical(read_notes(session))
assert REGRESSION_RESULT in read_source(session, failure_source)["content"]
assert len(session.state["windows"]) == 3
assert (
    len(canonical(search_history(session, "retry clock skew", limit=2, max_chars=1000)))
    <= 1000
)
assert len(canonical(bootstrap(session))) <= 5000
show(
    {
        "summary_has_exact_result": False,
        "initial_notes_have_exact_result": False,
        "original_source_has_exact_result": True,
        "bounded_retrieval": True,
    }
)

```json
{
  "summary_has_exact_result": false,
  "initial_notes_have_exact_result": false,
  "original_source_has_exact_result": true,
  "bounded_retrieval": true
}
```

In [38]:
foreign = MemorySession(
    provider,
    embeddings,
    {"memory_id": MEMORY_ID, "user_id": "different-user"},
    session.state_id,
    deepcopy(session.state),
)
try:
    read_source(foreign, failure_source)
    raise AssertionError("A foreign user must not read this source.")
except ValueError:
    print("Cross-user read refused, even with a copied source ID.")

try:
    bootstrap(session, max_chars=10)
    raise AssertionError("Oversized notes must not be silently dropped.")
except ValueError:
    print("Oversized note bootstrap refused explicitly.")

Cross-user read refused, even with a copied source ID.
Oversized note bootstrap refused explicitly.


## 11 · GPT-6 writes a next-step note

After recovering the evidence, the model can turn it into an actionable note.
We specify a new key and category for easy inspection, while leaving the wording
and source selection to GPT-6. Initial notes remain unchanged. Inspect whether
the new note proposes a useful fix while preserving the API, order, and attempt
limit; valid source IDs alone do not prove the recommendation is correct.

In [39]:
prompt = (
    "Use write_memory_note to save a concise actionable next step under key next-step, "
    "category next_step, expected_revision 0. Cite the original regression tool source "
    "you just read. Preserve the API, retry-order and four-attempt constraints. "
    "Leave existing notes unchanged, then explain the proposed next step."
)
note_answer, _ = run_turn(session, agent, prompt, bootstrap(session))
new_note = next(n for n in read_notes(session) if n["key"] == "next-step")
assert new_note["revision"] == 1 and new_note["category"] == "next_step"
assert failure_source in new_note["source_ids"]
assert (
    next(n for n in read_notes(session) if n["key"] == "api-contract")["revision"] == 2
)
assert next(n for n in read_notes(session) if n["key"] == "first-fix")["revision"] == 1
show({"answer": note_answer, "model_written_note": new_note})

```json
{
  "answer": "Saved `next-step` in category `next_step` using `expected_revision: 0`. Existing notes were left unchanged.\n\nProposed next step: apply the clock offset only once, then rerun `test_clock_skew` to confirm **7000 ms**, not **9000 ms**. Preserve the existing API, retry order, and maximum of four retry attempts.\n\nOriginal regression source: `b67f6260-1709-5450-a42e-e16e00b54c64` (`clock_offset_applied_twice`).",
  "model_written_note": {
    "category": "next_step",
    "content": "Fix retry-delay calculation so clock offset is applied only once (regression cause: clock_offset_applied_twice); rerun test_clock_skew and verify expected_delay_ms=7000 rather than actual_delay_ms=9000. Preserve the existing API and retry attempt order, with a maximum of four retry attempts.",
    "key": "next-step",
    "note_id": "68b5a620-c114-5005-928f-a3400a18ca65",
    "revision": 1,
    "source_ids": [
      "b67f6260-1709-5450-a42e-e16e00b54c64",
      "4ffd2826-7aaf-5153-a751-ae0e6d8912e1",
      "e4a197d3-3d99-50d6-8e5e-a39166d16d7d"
    ],
    "supersedes": null,
    "window": "window-3"
  }
}
```

## 12 · Gaps and improvements worth making to Memorizz

| Area | This standalone implementation | Package or research gap |
|---|---|---|
| Notes | Private records, individual revisions, source IDs, exact head lookup. | A public note-journal API could replace these notebook functions. Selection and stale-note handling need evaluation. |
| Tool capture | Explicitly captures our work-tool results before returning. | Unreleased opt-in `persist_all_results` support would generalize short-result retention. Arbitrary tools and exceptions need runtime capture hooks. |
| Search | Published vector search over source chunks, validated against canonical records. | Direct `search_tool_logs` support is not in 0.10.0. Production archives need ranking tests, pagination, and index repair. |
| Oracle reads | Uses shared-memory exact reads and metadata returned with search results. | Unreleased fixes for knowledge-base metadata and physical tool-log ID reads are not assumed installed. |
| GPT-6 | Published Responses adapter with explicit context size and output limit. | A release with GPT-6 model mappings would remove the explicit context-size setting. |
| Context coverage | Explicit user/final-assistant turns and the supplied work tools. | Every request, intermediate exchange, interrupted turn, external context payload, and hidden reasoning are not preserved here. |
| Rollover | Host-controlled boundary between completed turns. | Automatic pre-overflow handling inside long tool loops, with token-aware budgets. |
| Durability | Stable source IDs, writes before checkpoint advancement, one writer. | Cross-record transactions, concurrent writers, and orphan cleanup after crashes. |
| OpenAI equivalence | Reproduces the stated note-and-source behavior on this run. | Private policies, training, algorithms, and performance remain unspecified. |

### What to measure next

Use independent tasks with hidden facts and several forced boundaries. Vary the
omitted detail, correct requirements, and include tempting failed approaches.
Measure exact fact recovery, citation support, stale-note use, repeated failed
actions, isolation, restart recovery, prompt tokens, retrieval latency, and cost.
Compare summary plus source retrieval, notes only, and notes plus source retrieval
with equal tools and budgets. That comparison helps distinguish the value of
individual notes from the value of simply retaining an archive.

## 13 · Cleanup and recap

Cleanup removes only records belonging to this task/user scope and the journal's
recorded IDs. It does not drop Oracle tables or remove another task. Separate
Memorizz observability records follow their own retention policy. Interrupted
writes may leave orphan records; this small controller is not a general retention
service. Set `MEMORIZZ_CONTEXT_KEEP_DATA=1` before running to keep the lesson data.

The helper below checks each record's scope before deletion. Shared-memory reads
use the logical ID; deletion uses the returned physical row ID, which matters
for Oracle. Missing rows allow cleanup to resume after a partial cleanup.

In [40]:
def delete_record(session, identifier, kind):
    row = session.provider.retrieve_by_id(identifier, kind)
    if row is None:
        return
    if kind == MemoryType.SHARED_MEMORY:
        read_record(session, identifier)
    elif any(row.get(k) != v for k, v in session.scope.items()):
        raise ValueError("Refusing to delete an out-of-scope record.")
    if not session.provider.delete_by_id(record_id(row), kind):
        raise RuntimeError("Record deletion failed.")

In [41]:
def cleanup(session):
    for kind in (MemoryType.CONVERSATION_MEMORY, MemoryType.TOOL_LOG):
        for row in session.provider.list_all(kind, user_id=session.scope["user_id"]):
            if all(row.get(k) == v for k, v in session.scope.items()):
                if not session.provider.delete_by_id(record_id(row), kind):
                    raise RuntimeError("Runtime record deletion failed.")
    for identifier in session.state["index_ids"]:
        delete_record(session, identifier, MemoryType.KNOWLEDGE_BASE)
    note_ids = [i for revisions in session.state["notes"].values() for i in revisions]
    identifiers = session.state["sources"] + note_ids + session.state["checkpoints"]
    for identifier in [*identifiers, session.state_id]:
        delete_record(session, identifier, MemoryType.SHARED_MEMORY)

In [42]:
agent.close(close_memory_provider=False)
model.client.close()
if KEEP_DATA:
    show({"retained": True, **session.scope, "window": session.state["window"]})
else:
    cleanup(session)
    assert provider.retrieve_by_id(session.state_id, MemoryType.SHARED_MEMORY) is None
    print("Deleted this lesson's recorded memory rows.")
provider.close()
print(
    "Standalone lesson passed with real GPT-6, OpenAI embeddings, and published Memorizz."
)

Deleted this lesson's recorded memory rows.
Standalone lesson passed with real GPT-6, OpenAI embeddings, and published Memorizz.


### What to take away

Keep conclusions small and individually revisable. Keep original evidence
recoverable. Make the active prompt a bounded view of both. Notes support
continuity; source retrieval handles the questions those notes did not anticipate.

### References

- [OpenAI GPT-6 Astra announcement](https://openai.com/index/gpt-6-astra/): behavioral motivation from the supplied paragraph.
- [GPT-6 Astra API guide](https://developers.openai.com/api/docs/guides/latest-model): model, endpoint, and reasoning settings.
- [OpenAI embedding dimensions](https://developers.openai.com/api/docs/guides/embeddings#reducing-embedding-dimensions): real embeddings and supported shortening.
- [Memorizz 0.10.0 on PyPI](https://pypi.org/project/memorizz/0.10.0/): the package used by this notebook and its installation options.

To repeat the experiment, restart the kernel and run all cells. Re-running a
note-writing cell with a stale revision intentionally raises a conflict.